# 02 — Data Cleaning (Phase 2)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 2: import และเรียกใช้ฟังก์ชันจาก `src/rri/cleaning/`
เท่านั้น — ไม่มี business logic ในเซลล์ตาม CLAUDE.md N2 ลำดับขั้นตอนคงที่ตามที่ตกลงไว้:

1. Extract + structural fixes (encoding, BOM, header typo, twin-column coalesce, BE→CE)
2. Harmonize category taxonomy (**ก่อน** dedup)
3. Resolve dates ← **CHECKPOINT**
4. Deduplicate
5. Remaining field-level fixes (coordinates, times, injury policy, nulls)
6. Validate + write staging

อ้างอิงข้อเท็จจริงและเหตุผลทั้งหมดจาก Phase 1: [`CLAUDE.md`](../CLAUDE.md) §6 และ
[`docs/data_quality_findings.md`](../docs/data_quality_findings.md)

## 0. Setup

In [1]:
import sys
from datetime import UTC, datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import polars as pl

from config.settings import settings
from rri.cleaning.dates import (
    apply_date_certainty,
    check_year_match,
    day_of_month_histogram,
    parse_dates,
    resolve_transposed_pairs_by_likelihood,
)
from rri.cleaning.extract import (
    add_extract_metadata,
    coalesce_twin_columns,
    convert_buddhist_year,
    pivot_rainfall_leads,
    read_raw_csv,
    rename_headers,
)
from rri.cleaning.log import CleaningLogEntry, cleaning_log_to_frame
from rri.cleaning.taxonomy import apply_taxonomy, load_taxonomy_mapping
from rri.utils.logging import configure_logging
from rri.utils.profiling import (
    SOURCE_FILE_DATE_FORMAT,
    check_duplicate_date_string_consistency,
    compute_report_lag_days,
    find_conflicting_groups,
    find_transposed_date_string_pairs,
    report_lag_quantiles_by_group,
)

configure_logging()
settings.ensure_writable_dirs()
RUN_TIMESTAMP = datetime.now(UTC)
cleaning_log: list[CleaningLogEntry] = []
settings.raw_dir

PosixPath('/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/00_raw')

## 1. Extract + structural fixes

เริ่มจาก extract ไฟล์ CSV ทั้ง 6 ไฟล์ (ไฟล์ accidents หลักจะผ่านอีก 5 ขั้นตอนถัดไป
ส่วนอีก 5 ไฟล์จบที่ขั้นตอนนี้) ทุกไฟล์อ่านด้วย `read_raw_csv` (encoding ตาม P7.2 —
BOM 5 ใน 6 ไฟล์, `stt_car_fuel` ไม่มี) แล้วเติม `_source_file`/`_ingested_at`/`_row_hash`

In [2]:
accidents_raw = read_raw_csv(settings.raw_dir / "thailand_accidents_2019_2026_combined.csv")
accidents_raw = add_extract_metadata(
    accidents_raw, "thailand_accidents_2019_2026_combined.csv", RUN_TIMESTAMP
)

accidents_step1, twin_logs = coalesce_twin_columns(accidents_raw)

for log_entry in twin_logs:
    cleaning_log.append(
        CleaningLogEntry(
            step="1_extract",
            rule=f"coalesce_twin_columns:{log_entry.old_column}",
            rows_before=log_entry.total_rows,
            rows_affected=log_entry.rows_from_new,
            description=(
                f"{log_entry.old_column} <- {log_entry.new_column}: "
                f"{log_entry.rows_from_old} from old, {log_entry.rows_from_new} from new, "
                f"{log_entry.rows_still_null} still null"
            ),
        )
    )

cleaning_log_to_frame(cleaning_log)

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""1_extract""","""coalesce_twin_columns:รถยนต์นั…",161307,8738,"""รถยนต์นั่งส่วนบุคคล <- รถยนต์น…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกไม่เกิน10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกมากกว่า10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถอื่นๆ""",161307,8738,"""รถอื่นๆ <- อื่นๆ: 133375 from …"


**ยืนยันตรงกับ Phase 1**: overlap ระหว่างคอลัมน์เก่า/ใหม่ทั้ง 4 คู่ = 0 แถว (ถ้าไม่ตรง
`coalesce_twin_columns` จะ raise ทันที ไม่ใช่ coalesce มั่ว ๆ) จำนวนแถวที่มาจากคอลัมน์ใหม่ต่อคู่
ตรงกับสัดส่วน 5.4% ที่ Phase 1 พบ

In [3]:
rainfall_raw = read_raw_csv(settings.raw_dir / "thailand_rainfall_2021_2026_combined.csv")
rainfall_raw = add_extract_metadata(
    rainfall_raw, "thailand_rainfall_2021_2026_combined.csv", RUN_TIMESTAMP
)
rainfall_renamed = rename_headers(rainfall_raw, {"Foreast_Month": "Forecast_Month"})

rainfall_step1 = pivot_rainfall_leads(rainfall_renamed)
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "pivot_rainfall_leads",
        rainfall_renamed.height,
        rainfall_step1.height,
        f"{rainfall_renamed.height} rows (6 leads x group) -> "
        f"{rainfall_step1.height} groups x rain_lead0..5",
    )
)
pl.DataFrame(
    [
        {
            "rows_before_pivot": rainfall_renamed.height,
            "rows_after_pivot": rainfall_step1.height,
            "columns": rainfall_step1.columns,
        }
    ]
)

rows_before_pivot,rows_after_pivot,columns
i64,i64,list[str]
30954,5159,"[""Initial_Year"", ""Initial_Month"", … ""rain_lead5""]"


**ไม่มีแถวหายไป**: `rows_before_pivot / 6 == rows_after_pivot` (6 lead-time ต่อกลุ่มกลายเป็น
1 แถว x 6 คอลัมน์ ไม่ใช่การ dedup) ยืนยันตาม P7.5

In [4]:
registration_raw = read_raw_csv(settings.raw_dir / "car_registered_new_2565_2569_combined.csv")
registration_raw = add_extract_metadata(
    registration_raw, "car_registered_new_2565_2569_combined.csv", RUN_TIMESTAMP
)
registration_step1 = convert_buddhist_year(registration_raw, "ปี พ.ศ.")
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "convert_buddhist_year",
        registration_step1.height,
        registration_step1.height,
        "added year_ce = ปี พ.ศ. - 543 for every row (original column kept)",
    )
)
registration_step1.select("ปี พ.ศ.", "year_ce").unique().sort("ปี พ.ศ.")

ปี พ.ศ.,year_ce
str,i64
"""2565""",2022
"""2566""",2023
"""2567""",2024
"""2568""",2025
"""2569""",2026


**ตรวจแล้ว**: พ.ศ. 2565–2569 กลายเป็น ค.ศ. 2022–2026 ตรงตามสูตร `CE = BE - 543`
คอลัมน์เดิมยังอยู่ครบ (ไม่ทำลายข้อมูลดิบ)

In [5]:
flood_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_flood_risk_areas.csv"),
    "thailand_flood_risk_areas.csv",
    RUN_TIMESTAMP,
)
heavy_rain_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_heavy_rain_R35mm.csv"),
    "thailand_heavy_rain_R35mm.csv",
    RUN_TIMESTAMP,
)
stt_car_fuel_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "stt_car_fuel_at_25690228.csv"),
    "stt_car_fuel_at_25690228.csv",
    RUN_TIMESTAMP,
)

pl.DataFrame(
    [
        {"file": "flood_risk", "rows": flood_raw.height, "cols": flood_raw.width},
        {"file": "heavy_rain_r35mm", "rows": heavy_rain_raw.height, "cols": heavy_rain_raw.width},
        {"file": "stt_car_fuel", "rows": stt_car_fuel_raw.height, "cols": stt_car_fuel_raw.width},
    ]
)

file,rows,cols
str,i64,i64
"""flood_risk""",27023,16
"""heavy_rain_r35mm""",3465,7
"""stt_car_fuel""",4643,8


ไฟล์อีก 3 ไฟล์นี้ไม่มี structural fix เฉพาะใน Step 1 (นอกจาก encoding/metadata) —
เขียนลง staging ตรง ๆ

In [6]:
accidents_step1.write_parquet(settings.staging_dir / "accidents.parquet")
rainfall_step1.write_parquet(settings.staging_dir / "rainfall.parquet")
registration_step1.write_parquet(settings.staging_dir / "car_registered_new.parquet")
flood_raw.write_parquet(settings.staging_dir / "flood_risk.parquet")
heavy_rain_raw.write_parquet(settings.staging_dir / "heavy_rain_r35mm.parquet")
stt_car_fuel_raw.write_parquet(settings.staging_dir / "stt_car_fuel.parquet")

sorted(p.name for p in settings.staging_dir.glob("*.parquet"))

['accidents.parquet',
 'car_registered_new.parquet',
 'flood_risk.parquet',
 'heavy_rain_r35mm.parquet',
 'rainfall.parquet',
 'stt_car_fuel.parquet']

**Step 1 เสร็จสมบูรณ์** — เขียนไฟล์ทั้ง 6 ลง `data/10_staging/` แล้ว `accidents.parquet`
จะถูกเขียนทับอีกครั้งใน Step 6 หลังผ่าน Step 2–5 ครบ ไฟล์ที่เหลือจบที่นี่ (ไม่มีขั้นตอนถัดไปแตะต้อง)

## 2. Harmonize category taxonomy (ก่อน dedup)

โหลด `accidents.parquet` จาก Step 1 แล้ว apply mapping table ทั้ง 3 ตัว
(`config/mappings/{cause,crash_type,road_geometry}_groups.csv` — สร้างโดย
`scripts/build_taxonomy_mappings.py`, ตรวจสอบด้วยมือแล้ว) **ต้องทำก่อน Step 4 (dedup)**
เพราะ P5 พบว่า 74% ของ "conflict" ในคู่ duplicate ที่คอลัมน์ `บริเวณที่เกิดเหตุ` จริง ๆ
เป็นแค่คำพ้องความหมายคนละชุด ไม่ใช่ conflict จริง

In [7]:
accidents_step1 = pl.read_parquet(settings.staging_dir / "accidents.parquet")

cause_mapping = load_taxonomy_mapping(settings.mappings_dir / "cause_groups.csv")
crash_type_mapping = load_taxonomy_mapping(settings.mappings_dir / "crash_type_groups.csv")
road_geometry_mapping = load_taxonomy_mapping(settings.mappings_dir / "road_geometry_groups.csv")

accidents_step2, cause_log = apply_taxonomy(accidents_step1, "มูลเหตุสันนิษฐาน", cause_mapping)
accidents_step2, crash_type_log = apply_taxonomy(
    accidents_step2, "ลักษณะการเกิดเหตุ", crash_type_mapping
)
accidents_step2, geometry_log = apply_taxonomy(
    accidents_step2, "บริเวณที่เกิดเหตุ", road_geometry_mapping
)
accidents_step2, geometry_secondary_log = apply_taxonomy(
    accidents_step2,
    "บริเวณที่เกิดเหตุอื่น",
    road_geometry_mapping,
    new_column="บริเวณที่เกิดเหตุอื่น_group",
)

for taxonomy_log in (cause_log, crash_type_log, geometry_log, geometry_secondary_log):
    cleaning_log.append(
        CleaningLogEntry(
            "2_taxonomy",
            f"apply_taxonomy:{taxonomy_log.column}",
            taxonomy_log.total_rows,
            taxonomy_log.rows_mapped,
            f"mapped={taxonomy_log.rows_mapped} null={taxonomy_log.rows_null} "
            f"unmapped={taxonomy_log.rows_unmapped} {taxonomy_log.unmapped_values[:5]}",
        )
    )

pl.DataFrame(
    [
        {
            "column": log_entry.column,
            "rows_mapped": log_entry.rows_mapped,
            "rows_null": log_entry.rows_null,
            "rows_unmapped": log_entry.rows_unmapped,
            "unmapped_sample": log_entry.unmapped_values[:5],
        }
        for log_entry in (cause_log, crash_type_log, geometry_log, geometry_secondary_log)
    ]
)

column,rows_mapped,rows_null,rows_unmapped,unmapped_sample
str,i64,i64,i64,list[null]
"""มูลเหตุสันนิษฐาน""",158627,2680,0,[]
"""ลักษณะการเกิดเหตุ""",153244,8063,0,[]
"""บริเวณที่เกิดเหตุ""",147866,13441,0,[]
"""บริเวณที่เกิดเหตุอื่น""",8462,152845,0,[]


**ตรวจ**: `rows_unmapped` ต้องเป็น 0 ทุกคอลัมน์ (mapping table ครอบคลุมทุกค่าที่พบจริงใน
ข้อมูล ณ ตอนสร้างตาราง) ถ้าไม่เป็น 0 แปลว่าข้อมูลจริงมีค่าที่ mapping table ไม่รู้จัก — ต้อง
หยุดแล้วรายงาน ไม่ใช่เดาแล้วปล่อยผ่าน

In [8]:
geometry_conflicts_before = find_conflicting_groups(accidents_step2, ["ACC_CODE"], "บริเวณที่เกิดเหตุ")
geometry_conflicts_after = find_conflicting_groups(
    accidents_step2, ["ACC_CODE"], "บริเวณที่เกิดเหตุ_group"
)

dup_groups = (
    accidents_step2.filter(pl.col("ACC_CODE").is_not_null())
    .group_by("ACC_CODE")
    .len()
    .filter(pl.col("len") > 1)
)

pl.DataFrame(
    [
        {
            "duplicate_groups_checked": dup_groups.height,
            "conflicts_before_harmonization": len(geometry_conflicts_before),
            "conflict_rate_before_pct": round(
                len(geometry_conflicts_before) / dup_groups.height * 100, 1
            ),
            "conflicts_after_harmonization": len(geometry_conflicts_after),
            "conflict_rate_after_pct": round(
                len(geometry_conflicts_after) / dup_groups.height * 100, 1
            ),
        }
    ]
)

duplicate_groups_checked,conflicts_before_harmonization,conflict_rate_before_pct,conflicts_after_harmonization,conflict_rate_after_pct
i64,i64,f64,i64,f64
1641,1216,74.1,52,3.2


**เทียบกับ Phase 1**: conflict rate เดิม (raw `บริเวณที่เกิดเหตุ`) ควรตรงกับ P5 ที่ 74%
(1,216/1,641) หลัง harmonize ด้วย `บริเวณที่เกิดเหตุ_group` (group ตามประเภทถนน ตัด slope
descriptor ทิ้ง) ควรลดลงเหลือแค่ residual ที่เป็น conflict จริง (ประเภทถนนต่างกันจริง ไม่ใช่แค่
คำอธิบายความชันต่างกัน)

In [9]:
accidents_step2.write_parquet(settings.staging_dir / "accidents.parquet")
cleaning_log_to_frame(cleaning_log).filter(pl.col("step") == "2_taxonomy")

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""2_taxonomy""","""apply_taxonomy:มูลเหตุสันนิษฐา…",161307,158627,"""mapped=158627 null=2680 unmapp…"
"""2_taxonomy""","""apply_taxonomy:ลักษณะการเกิดเห…",161307,153244,"""mapped=153244 null=8063 unmapp…"
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,147866,"""mapped=147866 null=13441 unmap…"
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,8462,"""mapped=8462 null=152845 unmapp…"


**Step 2 เสร็จสมบูรณ์** — เขียนทับ `accidents.parquet` ด้วยเวอร์ชันที่มี taxonomy
harmonized แล้ว (`_group` columns เพิ่มเข้ามา คอลัมน์ raw เดิมยังอยู่ครบ) พร้อมสำหรับ Step 3

## 3. Resolve dates — CHECKPOINT

โหลด `accidents.parquet` จาก Step 2 แล้ว parse วันที่ตามรูปแบบต่อไฟล์ (P1) จากนั้นตรวจ
safety net ทั้ง 4 ข้อ และแก้ 448 คู่ transposed-date ด้วย **likelihood discriminator ต่อหน่วยงาน**
(แทนที่ binary threshold test เดิมของ Phase 1) — เทียบ lag ของแต่ละตัวเลือกกับค่ามัธยฐานของ
หน่วยงานนั้น เลือกตัวที่ใกล้กว่า แล้วบันทึก margin ไว้เป็นตัวชี้วัดความมั่นใจ

In [10]:
accidents_step2 = pl.read_parquet(settings.staging_dir / "accidents.parquet")
accidents_step3 = parse_dates(accidents_step2, SOURCE_FILE_DATE_FORMAT)

pl.DataFrame(
    [
        {
            "rows": accidents_step3.height,
            "accident_date_null": accidents_step3["accident_date"].null_count(),
            "report_date_null": accidents_step3["report_date"].null_count(),
        }
    ]
)

rows,accident_date_null,report_date_null
i64,i64,i64
161307,1,82


In [11]:
year_match = check_year_match(accidents_step3)

pl.DataFrame(
    [
        {
            "rows_checked": year_match.rows_checked,
            "rows_unparseable": year_match.rows_unparseable,
            "rows_mismatched": year_match.rows_mismatched,
        }
    ]
)

rows_checked,rows_unparseable,rows_mismatched
i64,i64,i64
161307,1,0


**Safety net #1**: ปีที่ parse ได้ต้องตรงกับคอลัมน์ `ปีที่เกิดเหตุ` เดิมทุกแถว — mismatch ต้องเป็น 0
(ถ้าไม่ใช่ 0 ต้องหยุดและรายงานทันที ไม่ใช่ปรับตัวเลขให้ผ่าน)

In [12]:
safety_net_4a = check_duplicate_date_string_consistency(
    accidents_step3, "ACC_CODE", "วันที่เกิดเหตุ", "accident_date"
)

pl.DataFrame(
    [
        {
            "groups_checked (identical raw string)": safety_net_4a.groups_checked,
            "mismatched_groups": safety_net_4a.mismatched_groups,
        }
    ]
)

groups_checked (identical raw string),mismatched_groups
i64,i64
1193,0


**Safety net #4a**: กลุ่ม duplicate ที่ raw string เหมือนกันเป๊ะต้อง parse ได้วันที่เดียวกันเสมอ —
mismatch ต้องเป็น 0 (ตรงกับที่ Phase 1 ยืนยันไว้แล้วที่ 1,193 กลุ่ม)

In [13]:
transposed_keys = find_transposed_date_string_pairs(
    accidents_step3.filter(pl.col("ACC_CODE").is_not_null()), "ACC_CODE", "วันที่เกิดเหตุ"
)

lagged = compute_report_lag_days(
    accidents_step3, "source_file", "วันที่เกิดเหตุ", "วันที่รายงาน", SOURCE_FILE_DATE_FORMAT
)
agency_lag_stats = report_lag_quantiles_by_group(lagged, "หน่วยงาน")
agency_median_lag = {row["หน่วยงาน"]: row["median_days"] for row in agency_lag_stats.to_dicts()}
agency_max_lag = {row["หน่วยงาน"]: float(row["max_days"]) for row in agency_lag_stats.to_dicts()}

likelihood_resolutions = resolve_transposed_pairs_by_likelihood(
    accidents_step3.filter(pl.col("ACC_CODE").is_not_null()),
    "ACC_CODE",
    transposed_keys,
    "วันที่เกิดเหตุ",
    "วันที่รายงาน",
    "หน่วยงาน",
    month_first=True,
    agency_median_lag_days=agency_median_lag,
    agency_max_lag_days=agency_max_lag,
)

resolved_count = sum(1 for r in likelihood_resolutions if r.outcome == "resolved")
ambiguous_count = sum(1 for r in likelihood_resolutions if r.outcome == "ambiguous")
neither_viable_count = sum(
    1 for r in likelihood_resolutions if r.outcome == "ambiguous" and r.winner_date is None
)
order_violation_count = sum(1 for r in likelihood_resolutions if r.had_order_violation)

pl.DataFrame(
    [
        {
            "transposed_pairs_found": len(transposed_keys),
            "resolved": resolved_count,
            "ambiguous": ambiguous_count,
            "sum": resolved_count + ambiguous_count,
            "ambiguous_both_hard_valid": ambiguous_count - neither_viable_count,
            "ambiguous_neither_hard_valid": neither_viable_count,
            "groups_with_an_order_violation_elimination": order_violation_count,
        }
    ]
)

transposed_pairs_found,resolved,ambiguous,sum,ambiguous_both_hard_valid,ambiguous_neither_hard_valid,groups_with_an_order_violation_elimination
i64,i64,i64,i64,i64,i64,i64
448,321,127,448,112,15,336


**แก้ไขแล้ว (owner-requested review): เดิม likelihood อย่างเดียวทำให้ 448/448 resolved 0 ambiguous — เป็นสัญญาณเตือน ไม่ใช่ชัยชนะ** เพราะไม่มี hard filter คัดคู่ที่ผิดกฎ `accident_date ≤ report_date` ออกก่อน ตอนนี้ใช้ 2 ขั้น: **ขั้น 1 (hard filter)** คัดคู่ที่ผิดกฎ chronology ออกก่อน (เทียบกับ **ทุก** วันที่รายงานที่พบในกลุ่ม ไม่ใช่แค่คู่ของแถวตัวเอง — เพราะวันที่รายงานเองก็สลับตำแหน่งได้เหมือนกัน ตาม P5) **ขั้น 2 (likelihood)** ใช้ตัดสินเฉพาะเมื่อเหลือ 2 ตัวเลือกหลัง hard filter เท่านั้น — และคู่แบบนั้นยังคง `date_certainty = ambiguous` เสมอ (แม้จะบันทึก winner ไว้อ้างอิงก็ตาม) เพราะเป็นการเดา ไม่ใช่ข้อสรุปที่บังคับได้ทางตรรกะ

In [14]:
accidents_step3 = apply_date_certainty(accidents_step3, likelihood_resolutions, "ACC_CODE")

accidents_step3["date_certainty"].value_counts()

date_certainty,count
str,u32
"""ambiguous""",596
"""certain""",160711


In [15]:
histogram = day_of_month_histogram(accidents_step3)
pl.DataFrame({"day": list(histogram.keys()), "count": list(histogram.values())})

day,count
i64,i64
1,7124
2,5682
3,5400
4,5610
5,4960
…,…
27,5177
28,5467
29,5630


**Safety net #3**: กระจายตัวของวันที่ (day-of-month) ควรค่อนข้างสม่ำเสมอตลอด 1-31 (มีลดลงบ้างที่
29-31 ตามธรรมชาติของจำนวนวันในเดือน) ถ้า parse ผิดจะเห็น spike ที่วัน 1-12 ชัดเจน (เพราะ MDY/DMY สลับกัน
จะโดนบีบเข้าช่วง 1-12 เท่านั้น)

In [16]:
sample_keys = [r.group_key for r in likelihood_resolutions if r.outcome == "resolved"][:5]

rows = []
for key in sample_keys:
    resolution = next(r for r in likelihood_resolutions if r.group_key == key)
    sub = accidents_step3.filter(pl.col("ACC_CODE") == key).select(
        "ACC_CODE", "วันที่เกิดเหตุ", "วันที่รายงาน", "หน่วยงาน"
    )
    for row in sub.to_dicts():
        rows.append(
            {
                "ACC_CODE": row["ACC_CODE"],
                "raw_date": row["วันที่เกิดเหตุ"],
                "report_date_raw": row["วันที่รายงาน"],
                "agency": row["หน่วยงาน"],
                "winner_date": resolution.winner_date,
                "outcome": resolution.outcome,
                "had_order_violation": resolution.had_order_violation,
            }
        )
pl.DataFrame(rows)

ACC_CODE,raw_date,report_date_raw,agency,winner_date,outcome,had_order_violation
str,str,str,str,date,str,bool
"""8018989""","""7/9/2023""","""7/9/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-07-09,"""resolved""",true
"""8018989""","""9/7/2023""","""9/7/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-07-09,"""resolved""",true
"""7684265""","""3/6/2023""","""3/7/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-03-06,"""resolved""",true
"""7684265""","""3/6/2023""","""3/7/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-03-06,"""resolved""",true
"""7684265""","""6/3/2023""","""7/3/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-03-06,"""resolved""",true
…,…,…,…,…,…,…
"""7542305""","""2/1/2023""","""3/1/2023""","""กรมทางหลวงชนบท""",2023-01-02,"""resolved""",true
"""8222871""","""10/11/2023""","""10/24/2023""","""กรมทางหลวงชนบท""",2023-10-11,"""resolved""",true
"""8222871""","""11/10/2023""","""10/24/2023""","""กรมทางหลวงชนบท""",2023-10-11,"""resolved""",true


**ตัวอย่างคู่ที่ยังเหลือ `ambiguous` (ทั้งสองตัวเลือกผ่าน hard filter จริง ไม่ใช่ discriminator เดา):**

In [17]:
ambiguous_sample_keys = [
    r.group_key
    for r in likelihood_resolutions
    if r.outcome == "ambiguous" and r.winner_date is not None
][:5]

rows = []
for key in ambiguous_sample_keys:
    resolution = next(r for r in likelihood_resolutions if r.group_key == key)
    sub = accidents_step3.filter(pl.col("ACC_CODE") == key).select(
        "ACC_CODE", "วันที่เกิดเหตุ", "วันที่รายงาน", "หน่วยงาน"
    )
    for row in sub.to_dicts():
        rows.append(
            {
                "ACC_CODE": row["ACC_CODE"],
                "raw_date": row["วันที่เกิดเหตุ"],
                "report_date_raw": row["วันที่รายงาน"],
                "agency": row["หน่วยงาน"],
                "likelihood_winner": resolution.winner_date,
                "margin_days": resolution.margin_days,
                "outcome": resolution.outcome,
            }
        )
pl.DataFrame(rows)

ACC_CODE,raw_date,report_date_raw,agency,likelihood_winner,margin_days,outcome
str,str,str,str,date,f64,str
"""7642092""","""3/5/2023""","""8/5/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-05-03,94.0,"""ambiguous"""
"""7642092""","""5/3/2023""","""5/8/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-05-03,94.0,"""ambiguous"""
"""8740246""","""2/12/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-12-02,293.0,"""ambiguous"""
"""8740246""","""12/2/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-12-02,293.0,"""ambiguous"""
"""7688412""","""6/7/2023""","""10/7/2023""","""กรมทางหลวงชนบท""",2023-07-06,118.0,"""ambiguous"""
"""7688412""","""7/6/2023""","""7/10/2023""","""กรมทางหลวงชนบท""",2023-07-06,118.0,"""ambiguous"""
"""8740318""","""9/10/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-10-09,29.0,"""ambiguous"""
"""8740318""","""10/9/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-10-09,29.0,"""ambiguous"""
"""7642114""","""4/5/2023""","""8/5/2023""","""กรมทางหลวงชนบท""",2023-05-04,118.0,"""ambiguous"""


ทั้งสองตัวเลือกผ่าน hard filter (accident ≤ report ทุกวันที่รายงานในกลุ่ม, gap ≤ ceiling) จริง —
`likelihood_winner` บันทึกไว้อ้างอิงเท่านั้น `date_certainty` ของทั้งสองแถวเป็น `ambiguous` เสมอ

In [18]:
accidents_step3.write_parquet(settings.staging_dir / "accidents.parquet")
"step3 written"

'step3 written'

## 4. Deduplicate

## 5. Remaining field-level fixes

## 6. Validate + write staging